The following has been adapted from Hailo's DFC Tutorials 1 and 2 (Parsing and Optimizing with DFC). It was run from a docker container setup using Hailo AI SoftwareSuite


In [ ]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

import torchvision as tv
import torch

import cv2

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

In [ ]:
chosen_hw_arch = "hailo8l"

The ONNX model below was created using the `ultralytics` yolo11n.pt pretrained model, which was finetuned on a customised visdrone dataset (single class) and exported using:
```model.export(format='onnx', opset=14)```
    
the onnx output model was copied to the docker container into /local/shared_with_docker/yolov11n_visdrone.onnx
    
 

In [ ]:
onnx_model_name = "yolo11n_visdrone"
onnx_path = "/local/shared_with_docker/yolo11n_visdrone_2class.onnx"

The endnodes below were taken from the [yolov11n.yaml](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/networks/yolov11n.yaml) on the hailo_model_zoo github. The finetuned (on VisDroneCustomClass) onnx model was opened in netron to double check these end nodes
```
- /model.23/cv2.0/cv2.0.2/Conv
- /model.23/cv3.0/cv3.0.2/Conv
- /model.23/cv2.1/cv2.1.2/Conv
- /model.23/cv3.1/cv3.1.2/Conv
- /model.23/cv2.2/cv2.2.2/Conv
- /model.23/cv3.2/cv3.2.2/Conv
```

In [ ]:
runner = ClientRunner(hw_arch=chosen_hw_arch)
hn, npz = runner.translate_onnx_model(
    onnx_path,
    onnx_model_name,
    start_node_names=["/model.0/conv/Conv"],
    end_node_names=["/model.23/cv2.0/cv2.0.2/Conv", 
                   "/model.23/cv3.0/cv3.0.2/Conv",
                   "/model.23/cv2.1/cv2.1.2/Conv",
                   "/model.23/cv3.1/cv3.1.2/Conv",
                   "/model.23/cv2.2/cv2.2.2/Conv",
                   "/model.23/cv3.2/cv3.2.2/Conv"],
    net_input_shapes={"/model.0/conv/Conv": [1, 3, 640, 640]},
)

In [ ]:
#save the parsed model
hailo_model_har_name = f"{onnx_model_name}_hailo_model_op14.har"
runner.save_har(hailo_model_har_name)

## Model Optimization

In [ ]:
def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

def resize_with_letterbox(image_path, target_shape, padding_value=(0, 0, 0)):
    """
    Resizes an image with letterboxing to fit the target size, preserving aspect ratio.
    
    Parameters:
        image_path (str): Path to the input image.
        target_shape (tuple): Target shape in NHWC format (batch_size, target_height, target_width, channels).
        padding_value (tuple): RGB values for padding (default is black padding).
        
    Returns:
        letterboxed_image (ndarray): The resized image with letterboxing.
        scale (float): Scaling ratio applied to the original image.
        pad_top (int): Padding applied to the top.
        pad_left (int): Padding applied to the left.
    """
    # Load the image from the given path
    image = cv2.imread(image_path)
    
    # Check if the image was loaded successfully
    if image is None:
        raise ValueError(f"Error: Unable to load image from path: {image_path}")
    
    # Convert the image from BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Get the original image dimensions (height, width, channels)
    h, w, c = image.shape
    
    # Extract target height and width from target_shape (NHWC format)
    target_height, target_width = target_shape[1], target_shape[2]
    
    # Calculate the scaling factors for width and height
    scale_x = target_width / w
    scale_y = target_height / h
    
    # Choose the smaller scale factor to preserve the aspect ratio
    scale = min(scale_x, scale_y)
    
    # Calculate the new dimensions based on the scaling factor
    new_w = int(w * scale)
    new_h = int(h * scale)

    # Resize the image to the new dimensions
    resized_image = cv2.resize(image, (new_w, new_h),interpolation=cv2.INTER_LINEAR)
    
    # Create a new image with the target size, filled with the padding value
    letterboxed_image = np.full((target_height, target_width, c), padding_value, dtype=np.uint8)
    
    # Compute the position where the resized image should be placed (padding)
    pad_top = (target_height - new_h) // 2
    pad_left = (target_width - new_w) // 2
    
    # Place the resized image onto the letterbox background
    letterboxed_image[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_image

    final_image = np.expand_dims(letterboxed_image, axis=0)
    
    # Return the letterboxed image, scaling ratio, and padding (top, left)
    return final_image

In [ ]:
# quantization needs calibration data, use the training data for this
cal_images_path = "../data/visdrone/train/images/" # use training data for calib
cal_images_list = [img_name for img_name in os.listdir(cal_images_path) if os.path.splitext(img_name)[1] == ".jpg"]
dataset_sz = 4100 
calib_dataset = np.zeros((dataset_sz, 640, 640, 3))

for idx, img_name in enumerate(sorted(cal_images_list)):
    if idx==dataset_sz:
        break
    #img = Image.open(os.path.join(cal_images_path, img_name)).convert('RGB')
#     img_preproc = preproc(img)
    imgpth = os.path.join(cal_images_path, img_name)
    img_preproc = resize_with_letterbox(imgpth, (1,640,640,3))
    calib_dataset[idx, :, :, :] = img_preproc
    
print(f"Calibration dataset size: {calib_dataset.shape}")

In [ ]:
#load our parsed HAR from the Parsing Tutorial
assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
runner = ClientRunner(har=hailo_model_har_name)

In [ ]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("/local/shared_with_docker/visdrone/yolov11_nms_config_visdrone.json", meta_arch=yolov8, engine=cpu)

model_optimization_config(calibration, batch_size=16, calibset_size=4000)
model_optimization_flavor(optimization_level=3)
post_quantization_optimization(finetune, policy=enabled, learning_rate=0.0001, epochs=8, dataset_size=4000)

allocator_param(width_splitter_defuse=disabled)
 """
#post_quantization_optimization(finetune, policy=enabled, learning_rate=0.0001, epochs=8, dataset_size=4000)



# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(alls)
#runner.optimize_full_precision()
runner.optimize(calib_dataset)

In [ ]:
model_name = "yolo11n_visdrone"
# fp_model_har_path = f"{model_name}_fp_opt_model_visdrone.har"
# runner.save_har(fp_model_har_path)

#quant_model_har_path = f"{model_name}_visdrone2class_quantized_lvl3_opt.har"
quant_model_har_path = "yolo11n_visdrone_quant_optlvl4_model.har"
runner.save_har(quant_model_har_path)

In [ ]:
# load the model
runner = ClientRunner(har=quant_model_har_path, hw_arch=chosen_hw_arch)

# fp_opt_model_har_path = 'yolo11n_visdrone_fp_opt_model_visdrone.har'
# runner = ClientRunner(har=fp_opt_model_har_path, hw_arch=chosen_hw_arch)

## Model Inference and Evaluation

In [ ]:
!pwd

In [ ]:
# get a list of images (filename) and the corresponding image_id from annotations file

annotations_file = '/local/shared_with_docker/visdrone/annotations_VisDroneHumans_val.json'
images_path = "/local/shared_with_docker/visdrone/VisDrone2019-DET-val/images"

with open(annotations_file, 'r') as f:
    val_gt = json.load(f)
    f.close()
    
images = val_gt['images']
image_list = [(x['file_name'],x['id']) for x in images] 

    


In [ ]:
def run_inference(runner, sdk_type, input_data ):
    if sdk_type == 'quantized':
        with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
            output = runner.infer(ctx, input_data)
    else:
        with runner.infer_context(InferenceContext.SDK_FP_OPTIMIZED) as ctx:
            output = runner.infer(ctx, input_data)
    return output  

def remove_zero_padding(output, imgid):
    # remove zero-padding and transpose detections into single array
    combined = np.empty((7, 0)) # 1 classid, xywh, score

    for i in range(output.shape[1]):
#         print(f"processing class {i} for image {imgid}...")
        valid_mask = np.any(output[0,i,:,:] != 0, axis=(0))  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding

        dets = output[0, i, :, :last_valid_indices]
        class_col = np.full_like(dets[0,None], i) # add the classID into the array
        imgid_col = np.full_like(dets[0,None], imgid)
        dets = np.vstack((dets, class_col, imgid_col))
        combined = np.concatenate((combined, dets), axis=1)
#        print(f"\tfound {dets.shape[1]} detections for class {i}...")

    dets = combined.T
    return dets


def swap_columns(detections):
    detections[:, [0, 1,2,3]] = detections[:, [1, 0, 3, 2]]
    return detections


def ltxy2xywh(xywh):
    xywh[:,0] = xywh[:,0] # x
    xywh[:,1] = xywh[:,1] # y
    xywh[:,2] = xywh[:,2] - xywh[:,0] # x2 - x1
    xywh[:,3] = xywh[:,3] - xywh[:,1] # y2 - y1

    return xywh

def rescale_bbox(detections, w, h):
    w_arr = np.full_like(detections[:,0], w)
    h_arr = np.full_like(detections[:,0], h)
    
    detections[:,0] = detections[:,0] * w_arr
    detections[:,1] = detections[:,1] * h_arr
    detections[:,2] = detections[:,2] * w_arr
    detections[:,3] = detections[:,3] * h_arr
    
    return detections


def get_image_metadata(images_dict, img_id):
    for img in images_dict:
        if img['id'] == img_id:
            w, h = img['width'], img['height']
            name = img["file_name"]

    return w,h, name


In [ ]:
# create an np array for the validation dataset
dataset_sz = len(image_list)
val_dataset = np.zeros((dataset_sz, 640, 640, 3))
val_imageids = np.zeros((dataset_sz,1))
for idx, imagename_id in enumerate(image_list):
    imgname, imgid = imagename_id
    image_file = os.path.join(images_path, imgname)
    if idx==dataset_sz:
        break
    img = Image.open(image_file).convert('RGB')
    img_preproc = preproc(img)
    val_dataset[idx, :, :, :] = img_preproc
    val_imageids[idx] = imgid

In [ ]:
def detections_batch_generation(dataset, dataset_id, images_dict):
    batch_dets = run_inference(runner=runner, sdk_type='quantized', input_data=dataset)
    all_dets = []
    for idx, output in enumerate(batch_dets):
        imgid = dataset_id[idx]
        
        # postprocessing steps
        # remove zero padding
        output = output.reshape(1,2,5,100) # expects a batch
        detections = remove_zero_padding(output, imgid)
        
        # convert to xywh
        xywh = np.full_like(detections, detections)
        xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
        xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h
        
        # rescale detections
        w,h, name = get_image_metadata(images_dict, imgid) # images 
        dets_scaled = rescale_bbox(xywh, w, h)
        
        all_dets.append(dets_scaled)
    return(all_dets)


def batch_detections_to_json(batch):
    class_remap = {0: 1, 1: 2}
    res = []
    annId = 1
    for detections in batch:
        for el in detections:
            x,y,w,h,conf,classid,imgid = el
            imgid = int(imgid)
            res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
            annId +=1
    return res


def save_json_file(filename, data):
    with open(outputfile, 'w') as outf:
        json.dump(data, outf)
        outf.close()

    print(f"Total of {len(processed_outputs)} annotations saved to {outputfile}")

In [ ]:
batch_detections = detections_batch_generation(val_dataset, val_imageids, images)
processed_outputs = batch_detections_to_json(batch_detections)

outputfile = f"/local/shared_with_docker/visdrone/detections_VisDrone_val_quant_optlvl4.json" 
save_json_file(outputfile, processed_outputs)

## Compile

In [ ]:
quant_model_har_path = "yolo11n_visdrone_quant_opt_model_visdrone.har"
runner = ClientRunner(har=quant_model_har_path, hw_arch=chosen_hw_arch)

hef = runner.compile()

file_name = f"{model_name}.hef"
with open(file_name, "wb") as f:
    f.write(hef)

## Redundant functions and code
This was used for single image inference, e.g. for iterating over each image

In [ ]:
# these functions are used for single image evaluation
# intented for use when iterating over the dataset
def get_input_data(image_dataset, image_id):
    assert image_id >0, f"Image id should be >0 as per coco formatting"

    imgname, imgid = image_dataset[image_id-1]
    imgfile = os.path.join(images_path, imgname)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, imgid, img

def get_single_image(image_rootpath, image_file):
    imgfile = os.path.join(image_rootpath, imgname)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, img

def detections_to_json(detections):
    class_remap = {0: 1, 1: 2}
    res = []
    annId = 1
    for el in detections:
        x,y,w,h,conf,classid,imgid = el
        imgid = int(imgid)
        res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
        annId +=1
    return res

def get_single_image(image_rootpath, image_file):
    imgfile = os.path.join(image_rootpath, image_file)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, img

def generate_detections_file(image_list, image_dir):
    all_dets = []
    for idx, imgdata in enumerate(image_list):
        imgname, imgid = imgdata
        img_input, img = get_single_image(image_dir, imgname)
        
        output = run_inference(runner=runner, sdk_type='x', input_data=img_input)
        detections = remove_zero_padding(output, imgid)

        xywh = np.full_like(detections, detections)
        xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
        xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h


        w,h, name = get_image_metadata(images, imgid)

        dets_scaled = rescale_bbox(xywh, w, h)
        all_dets.append(dets_scaled)
    return(all_dets)
# full_data_dets = generate_detections_file(image_list, images_path)

    

# img_input, imgid, img = get_input_data(image_list, 1)
# output = run_inference(runner=runner, sdk_type='opt', input_data=img_input)
# detections = remove_zero_padding(output, imgid)

# xywh = np.full_like(detections, detections)
# xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
# xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h


# w,h, name = get_image_metadata(images, imgid)

# dets_scaled = rescale_bbox(xywh, w, h)

# processed_outputs = detections_to_json(dets_scaled)
# outputfile = f"/local/shared_with_docker/visdrone/detections_VisDrone_quant_dets.json" 
# with open(outputfile, 'w') as outf:
#     json.dump(processed_outputs, outf)
#     outf.close()
    
# print(f"Total of {len(processed_outputs)} annotations saved to {outputfile}")